In [ ]:
# NOTEBOOK NAME
# SimpleVerticalPlots.ipynb
# NOTEBOOK NAME

# OPENING IMPORTS
import numpy as np
from matplotlib import pyplot as plt
import xarray as xr

from pathlib import Path      # used to play with pathnames to save

from PIL import Image         # used for creating gif loops
import os                     # used for retrieving file names

# # for projecting radar coordinates to lat and lon
# from pyproj import Geod

# # mapping things
import cartopy.crs as ccrs
# import cartopy.feature as cfeature
# from cartopy.io import shapereader

# # for adding lat/lon gridlines on plots
# import matplotlib.ticker as mticker
# from cartopy.mpl.gridliner import LATITUDE_FORMATTER, LONGITUDE_FORMATTER

# # SPECIAL METHOD TO IMPORT CUSTOM FUNCTIONS AND ELEVATION FROM LOCAL DIRECTORY
import sys
sys.path.append('/home/563/sg3241/Notebooks/PhD/CustomFunctions')
from CustomFunctions1 import *

# # # for adding a colourful topo base map to the CAPI plots
from custom_elevation import fetch_srtm, fetch_gebco_local
from matplotlib.colors import LinearSegmentedColormap
from matplotlib.colors import ListedColormap, BoundaryNorm, Normalize

In [ ]:
# Load in a DEM (update path and variable name as needed)
DEMpath = '/home/563/sg3241/QueenslandElevationGEBCO.nc'  # <- your DEM file
DEMdata = xr.open_dataset(DEMpath)
DEMelev = DEMdata['elevation']  # adjust if your var has a different name

In [ ]:
# CHAD FUNCTIONS TO CALCULATE DISTANCE FROM RADAR BASED ON HEIGHT ABOVE EARTH

def beam_height_curved(s_km, elev_deg, ke=4/3, re_km=6371.0):
    """
    Calculate radar beam centre height above Earth's surface accounting
    for Earth curvature and standard atmospheric refraction.
    Uses the effective Earth radius (4/3 Earth radius) model.

    Parameters
    ----------
    s_km     : float or array  — horizontal ground distance from radar [km]
    elev_deg : float           — radar elevation angle [degrees]
    ke       : float           — effective Earth radius factor (default 4/3)
    re_km    : float           — Earth radius [km] (default 6371)

    Returns
    -------
    height_km : float or array — beam centre height above surface [km]
    """
    theta = np.radians(elev_deg)
    ker   = ke * re_km

    pretend_height_km = s_km * np.tan(theta) # estimated height above radar without earth curve

    slant_km = np.sqrt(s_km**2 + pretend_height_km **2) # estimated slant distance from radar
    
    height_km = np.sqrt(slant_km**2 + ker**2 + 2 * slant_km * ker * np.sin(theta)) - ker # new estimated height with curvature
    return height_km

# LETS PRETEND FOR NOW THAT THE CURVATURE OF THE EARTH HAS NO EFFECT ON THE HORIZONTAL DISTANCE
# def slant_range_curved(s_km, elev_deg, ke=4/3, re_km=6371.0):
#     """
#     Calculate the true slant range from the radar to a point at horizontal
#     ground distance s_km, accounting for Earth curvature.
#     Used for minimum detectable reflectivity calculations.

#     Parameters
#     ----------
#     s_km     : float or array  — horizontal ground distance from radar [km]
#     elev_deg : float           — radar elevation angle [degrees] (not used in
#                                  range-only MDR but kept for completeness)
#     ke       : float           — effective Earth radius factor (default 4/3)
#     re_km    : float           — Earth radius [km] (default 6371)

#     Returns
#     -------
#     slant_km : float or array  — slant range from radar [km]
#     """
#     theta = np.radians(elev_deg)
#     ker   = ke * re_km
#     # slant range from geometry
#     slant_km = ker * np.arcsin(s_km * np.cos(theta) / 
#                                np.sqrt(s_km**2 + ker**2 + 2 * s_km * ker * np.sin(theta)))
#     return slant_km


def calculate_mdr(distance_km):
    """
    Minimum detectable reflectivity as a function of slant range.
    5 dBZ at 80 km, +6 dBZ per doubling of distance.

    Parameters
    ----------
    distance_km : float or array — slant range from radar [km]

    Returns
    -------
    mdr : float or array — minimum detectable reflectivity [dBZ]
    """
    return 5.0 + 6.0 * np.log2(distance_km / 80.0)


In [ ]:
# VERTICAL CROSS SECTION PLOTTING
# (LOADS IN FROM NET CDF FILES STORED IN SCRATCH)

# CHOOSE THE RADAR
RadarIDno = '22'  # 22 is Mackay, 106 is Townsville, 66 is Mt Staplyton (Brisbane) , 50 is Marabong (near Bris)

# CHOOSE THE DATE
# the day in consideration (YYYYMMDD) and time (hhmmss)       ALL IN UTC !!!
RadarYear  = 2024
RadarMonth = 3
RadarDay   = 9
# CHOOSE THE MINUTES YOU WANT TO LOOP OVER (5-MIN PERIODS) (INCLUSIVE OF START AND END TIMES)
LoopStartTime = '12:00'
LoopEndTime   = '12:05'

# CHOOSE THE SLICE OF THE CROSS-SECTION
EWsliceKM = 25 # [km]     # number of km north or south of the radar you want to take the east-west slice for x-section
EWsliceNorS = 'North'    # direction ['North' or 'South'] from the radar you want the slice taken

# CHOOSE YOUR ALTITUDE RANGE
MinHeight = 0 # [km] minimum height in plot
MaxHeight = 15 # [km] maximum height in the plot

# PLEASE MAKE IT SO THE PLOT VARIABLES CAN BE CHOSEN HERE!!!! (ADD STRING EXECUTERS AND SUCH)
# CHOOSE THE VARIABLE TO PLOT
Var  = 'CC'

# LIST OF POSSIBLE VARIABLES
# [Z]         'corrected_reflectivity'
# [CC]        'corrected_cross_correlation_ratio'
# [ZDR]       'corrected_differential_reflectivity'
# [KDP]       'corrected_specific_differential_phase'
# [PhiDP]     'corrected_differential_phase'
# [IntAtt]    'path_integrated_attenuation'
# [DifIntAtt] 'path_integrated_differential_attenuation'
# [EchClas]   'radar_echo_classification'
# [V]         'corrected_velocity'
# [AzSh]      'azshear'



# CHOOSE YOUR QUALITY CONTROL SETTINGS
# # taken from Aragon et al. 2024
# MinValidZDR = -4 # ZDR DOESNT WORK FOR MACKAY EARLY 2024 BECAUSE OF THE SOURCE RADAR DATA IN STORAGE
# MaxValidZDR =  4
MinValidRhoHV = 0


# CHOOSE YOUR SOUNDING DATA
Ztime = '00'
UpperAirSiteID = '95282'
# 95282 for Townsville, 
# 94299 for Willis Island



# USER CHOICE FOLLOW-ON SECTION

# radar choice follow-on
if (RadarIDno == '22'):
    RadarSiteName = 'Mackay'
elif (RadarIDno == '106'):
    RadarSiteName = 'Townsville'
elif (RadarIDno == '66'):
    RadarSiteName = 'Mt Staplyton'
elif (RadarIDno == '50'):
    RadarSiteName = 'Marabong'
else:
    RadarSiteName = 'Site ' + RadarIDno


# date choice follow-on
# add leading zeros for strings
YYYY = str(RadarYear).zfill(4)
MM = str(RadarMonth).zfill(2)
DD = str(RadarDay).zfill(2)
# write out the data in one string with and without dashes
RadarFileDate  = YYYY + MM + DD
RadarFileDatePrint = YYYY + '-' + MM + '-' + DD

# sounding data follow-on and loading
SoundingFolder = '/home/563/sg3241/TownsvilleSoundings/' 
SoundingFileName = RadarFileDate + Ztime + '-' + UpperAirSiteID + '.csv'
SoundingPath = SoundingFolder + SoundingFileName

SoundingData = pd.read_csv(SoundingPath)

# slice choice follow-on
# THIS CODE ASSUMES THE X AND Y GRIDS ARE EVERY 1 KM
# IT WILL HAVE TO BE REWRITTEN TO CONSIDER IN-BETWEEN VALUES
# THIS CODE ASSUMES THE X AND Y GRIDS ARE EVERY 1 KM
# IT WILL HAVE TO BE REWRITTEN TO CONSIDER IN-BETWEEN VALUES
Slice    = str(EWsliceKM) + 'kmEW'
# add a positive or negative sign to the slice for math
if (EWsliceNorS == 'South'):
    EWsliceKMsign = EWsliceKM * -1
elif (EWsliceNorS == 'North'):
    EWsliceKMsign = EWsliceKM * 1
else:
    print("Please choose EWsliceNorS to be 'North' or 'South'.")


# variable choice follow-on
if (Var == 'Z'):
    VarName     = 'Reflectivity'
    VarNameLong = 'corrected_reflectivity'
    VarMinVal =  -9 # [dBZ]
    VarMaxVal =   30 # [dBZ]
    VarUnit   = 'dBZ'
    VarColourBar = make_ChadMapZ()
    # fix the colour bar to all values no matter which range you choose to view
    VarColourBar_min = -9.0
    VarColourBar_max = 30.0
    VarTickSpacing = 3
    VarColourBar_norm = Normalize(vmin=VarColourBar_min, vmax=VarColourBar_max)
    
elif (Var == 'ZDR'):
    VarName     = 'Differential Reflectivity'
    VarNameLong = 'corrected_differential_reflectivity'
    VarMinVal = -5 # [dB]
    VarMaxVal =  5 # [dB]
    VarUnit   = 'dB'
    VarColourBar = 'RdBu'
elif (Var == 'CC'):
    VarName     = 'Correlation Coefficient'
    VarNameLong = 'corrected_cross_correlation_ratio'
    VarMinVal = 0.8 # [0 to 1]
    VarMaxVal = 1.0 # [0 to 1]
    VarUnit   = '-0 to 1'
    VarColourBar = 'nipy_spectral'
    # fix the colour bar to all values no matter which range you choose to view
    VarColourBar_min = 0.8
    VarColourBar_max = 1.0
    VarTickSpacing = 0.05
    VarColourBar_norm = Normalize(vmin=VarColourBar_min, vmax=VarColourBar_max)
elif (Var == 'KDP'):
    VarName     = 'Specific Differential Phase'
    VarNameLong = 'corrected_specific_differential_phase'
    VarMinVal = 0  # [deg/ km]
    VarMaxVal = 10 # [deg / km]
    VarUnit   = 'deg / km'
    VarColourBar = 'nipy_spectral'
elif (Var == 'PhiDP'):
    VarName     = 'Differential Phase'
    VarNameLong = 'corrected_differential_phase'
    VarMinVal = 0  # [deg]
    VarMaxVal = 30 # [deg]
    VarUnit   = 'deg'
    VarColourBar = 'nipy_spectral'
elif (Var == 'V'):
    VarName     = 'Velocity'
    VarNameLong = 'corrected_velocity'
    VarMinVal = -5 # [m/s]
    VarMaxVal =  5 # [m/s]
    VarUnit   = 'm/s'
    VarColourBar = 'RdYlGn_r'
    # fix the colour bar to all values no matter which range you choose to view
    VarColourBar_min = -5.0
    VarColourBar_max = 5.0
    VarTickSpacing = 1
    VarColourBar_norm = Normalize(vmin=VarColourBar_min, vmax=VarColourBar_max)
else:
    raise ValueError("Input Variable '" + Var + "' not available\n" + "Please choose from the following list:\n" + \
          "[Z] 'corrected_reflectivity', [CC] 'corrected_cross_correlation_ratio', [ZDR] 'corrected_differential_reflectivity'\n" + \
          "[KDP] 'corrected_specific_differential_phase', [PhiDP] 'corrected_differential_phase'")

    

# remember these plots are vertical cross sections
PlotType = 'Vert'
# LOOP OVER EVERY 5 MIN PERIOD IN THE DAY
# Parse start and end times
StartHour, StartMin = int(LoopStartTime.split(':')[0]), int(LoopStartTime.split(':')[1])
EndHour, EndMin     =   int(LoopEndTime.split(':')[0]),   int(LoopEndTime.split(':')[1])

# Convert to total minutes for easy comparison
StartMinOfDay = StartHour * 60 + StartMin
EndMinOfDay = EndHour * 60 + EndMin

for MinOfDay in range(StartMinOfDay, EndMinOfDay + 1, 5):
    # find the old indicies with which this code was written (0-23 for hours, 0-11 for 5-minute periods within hours)
    houri = MinOfDay // 60
    mini  = MinOfDay % 60
    
    RadarFileTime = str(houri).zfill(2) + str(mini).zfill(2) + '00' # write out the time in 6 digits (like 012040 for 01:20:40 AM)
    RadarFileTimePrint = str(RadarFileTime)[0:2] + ':' + str(RadarFileTime)[2:4] + ':' + str(RadarFileTime)[4:6] 
    
    # add a string of format hh:mm:ss for printing
    print('working on ' + RadarFileTimePrint)

    NetCDFstoragePath = ('/scratch/v46/sg3241/tmp/NetCDFs/CompressedRadarGrids/' + RadarIDno + '/' + YYYY + '/' + MM + '/' + DD + '/' + \
                            RadarIDno + '_' + RadarFileDate + '_' + RadarFileTime + '.nc')

    # try to load in the netcdf file and if it doesn't work, just keep going through the loop
    try:
        xgrid = xr.open_dataset(NetCDFstoragePath)
    except FileNotFoundError:
        print(f'File missing for {RadarFileTimePrint}, skipping: {NetCDFstoragePath}')
        continue

    # ROUGH QUALITY CONTROL SECTION
    # create a mask only where these quality control condtions are met
    ConditionGridA = xgrid['corrected_cross_correlation_ratio'] > MinValidRhoHV   # (y, x) boolean masks
    
    # I WOULD LIKE TO ADD CONDITIONS WITH DIFFERENTIAL REFLECTIVITY, BUT THIS RADAR DOES NOT HAVE VALID DATA YET
    # ConditionGridB = xgrid['corrected_differential_reflectivity'] > MinValidZDR
    # ConditionGridC = xgrid['corrected_differential_reflectivity'] < MaxValidZDR  
    
    ConditionGrid = ConditionGridA #* ConditionGridB * ConditionGridC   # combined boolean mask
    
    # THIS CODE ASSUMES THE X AND Y GRIDS ARE EVERY 1 KM
    # IT WILL HAVE TO BE REWRITTEN TO CONSIDER IN-BETWEEN VALUES

    # Coordinates in km (range from radar)
    yvalsKM = np.array(xgrid.x) * 0.001  # x coordinates in km
    # farthest south and north y-values
    MinSliceKM = int(np.min(yvalsKM))
    MaxSliceKM = int(np.max(yvalsKM))

    # find the vertical cross section index in the coordinates
    EWslicei = np.where(yvalsKM == EWsliceKMsign)[0][0]  # index in the x-coordinates where that north-south km value lives

    # EXPERIMENTAL TOPOGRAPHY SECTION
    # EXPERIMENTAL TOPOGRAPHY SECTION
    # EXPERIMENTAL TOPOGRAPHY SECTION

    # lat/lon fields in xgrid: dimensions (y, x)
    # Slice at the chosen north–south index EWslicei along x
    lat_slice = xgrid['lat'][EWslicei, :].values  # shape (nx,)
    lon_slice = xgrid['lon'][EWslicei, :].values  # shape (nx,)

    # Build DataArrays for interpolation
    LONdata = xr.DataArray(lon_slice, dims=('x',))
    LATdata = xr.DataArray(lat_slice, dims=('x',))

    # Interpolate DEM to the cross-section line
    dem_slice = DEMelev.interp(lon=LONdata, lat=LATdata)

    # Elevation in metres along the line
    TerrainSlice = dem_slice.values

    # For plotting in km, and do not let negative (ocean) go below 0
    TerrainSliceKM = np.maximum(TerrainSlice, 0.0) * 0.001

    Xkm = xgrid.x * 0.001  # east–west distance [km]

    # EXPERIMENTAL TOPOGRAPHY SECTION END
    # EXPERIMENTAL TOPOGRAPHY SECTION END
    # EXPERIMENTAL TOPOGRAPHY SECTION END

    # MINIMUM DETECTABLE REFLECTIVITY SECTION
    # MINIMUM DETECTABLE REFLECTIVITY SECTION
    # MINIMUM DETECTABLE REFLECTIVITY SECTION

    x_vals   = xgrid.x.values * 0.001  # east-west distance from radar [km]
    z_vals   = xgrid.z.values * 0.001  # height above surface [km]

    # Horizontal ground distance from radar at each x point along the slice
    horz_vals = np.sqrt(x_vals**2 + EWsliceKM**2)  # shape (nx,)

    # True slant range to each grid cell
    distance_grid = np.sqrt(horz_vals[np.newaxis, :]**2 + z_vals[:, np.newaxis]**2)  # shape (nz, nx)

    # Calculate MDR for each point in the grid
    mdr_grid = calculate_mdr(distance_grid)

    # MINIMUM DETECTABLE REFLECTIVITY SECTION END
    # MINIMUM DETECTABLE REFLECTIVITY SECTION END
    # MINIMUM DETECTABLE REFLECTIVITY SECTION END

    # RADAR BEAM CENTRE LINES SECTION
    # RADAR BEAM CENTRE LINES SECTION
    # RADAR BEAM CENTRE LINES SECTION

    # Elevation angles of the radar beams [degrees]
    elevation_angles = np.array([0.5, 0.8, 1.4, 2.4, 3.5, 4.7, 6.0, 7.8, 10, 13, 17, 23, 32])

    # Calculate beam centre heights using curved Earth model
    # beam_altitude_curves shape: (n_angles, nx)
    beam_altitude_curves = np.array([
        beam_height_curved(horz_vals, elev_angle)
        for elev_angle in elevation_angles
    ])

    # RADAR BEAM CENTRE LINES SECTION END
    # RADAR BEAM CENTRE LINES SECTION END
    # RADAR BEAM CENTRE LINES SECTION END

    # apply the condtional mask to the variable array before plotting
    ValidVariableArray = xgrid[VarNameLong].where(ConditionGrid)
    PlottingArray = ValidVariableArray[0,:,EWslicei,:]


    # REAL DATA PLOTTING
    # REAL DATA PLOTTING
    # REAL DATA PLOTTING
    fig, ax = plt.subplots(figsize=(25, 3))
    GridViewer = pcolormeshC(lon_slice, xgrid.z * 0.001, PlottingArray, ax=ax, 
                             cmap=VarColourBar, norm=VarColourBar_norm)
    
    # Plot minimum detectable reflectivity contours
    # Thin lines only for levels NOT divisible by 5
    mdr_levels_thin_only = [level for level in mdr_levels_thin if level % 5 != 0]
    contours_thin = ax.contour(lon_slice, z_vals, mdr_grid, levels=mdr_levels_thin_only, 
                               colors='black', linewidths=0.2, alpha=0.4, zorder=3,
                               linestyles='dashed')

    
    # Thick lines only for levels divisible by 5
    mdr_levels_thick = [level for level in mdr_levels_thin if level % 5 == 0]
    contours_thick = ax.contour(lon_slice, z_vals, mdr_grid, levels=mdr_levels_thick, 
                                colors='black', linewidths=0.4, alpha=0.7, zorder=3,
                                linestyles='dashed')

    
    # Labels on thin lines — small and subtle
    ax.clabel(contours_thin, inline=True, fontsize=6, fmt='%1.0f', 
              inline_spacing=2)
    
    # Labels on thick lines — slightly larger
    ax.clabel(contours_thick, inline=True, fontsize=8, fmt='%1.0f dBZ', 
              inline_spacing=2)

    
    # Plot radar beam centre lines as curves across the slice
    for i, elev_angle in enumerate(elevation_angles):
        beam_curve = beam_altitude_curves[i, :]  # altitude [km] at each x point along the slice
    
        # Only plot if any part of the beam is within the plot range
        if np.any(beam_curve <= MaxHeight + 3):
            ax.plot(lon_slice, beam_curve, color=[0.5, 0.5, 1.0], linewidth=1.8,
                    alpha=0.3, linestyle='-', zorder=2)
    
            # Find a sensible x position for the label — use the rightmost point still in range
            in_range_mask = beam_curve <= MaxHeight + 3
            label_idx = np.where(in_range_mask)[0][-1]  # rightmost in-range index
    
            ax.text(lon_slice[label_idx], beam_curve[label_idx], f'{elev_angle}°',
                    fontsize=7, va='bottom', ha='center',
                    bbox=dict(boxstyle='round,pad=0.2', facecolor='white', alpha=0.7, edgecolor='none'))

    # Plot the line showing ground/topography
    ax.plot(lon_slice, TerrainSliceKM, color=[0.3, 0.3, 0.3], linewidth=0.8, zorder=5)
    ax.fill_between(lon_slice, 0, TerrainSliceKM, color=[0.3, 0.3, 0.3], zorder=4)
    
    cbar = plt.colorbar(GridViewer, ax=ax, label=VarName + ' [' + VarUnit + ']', pad=0.01)

    cbar.ax.set_ylim(VarMinVal, VarMaxVal)
    cbar.set_ticks(np.arange(VarMinVal, VarMaxVal+1, VarTickSpacing))
    
    # ADD A FUNCTION THAT CAN FIND THE TROPOPAUSE !!!
    # ADD A FUNCTION THAT CAN FIND THE TROPOPAUSE !!!
    # ADD A FUNCTION THAT CAN FIND THE TROPOPAUSE !!!
    TropopauseAlt = 16.8
    # plot the tropopause
    ax.axhline(y=TropopauseAlt, color=[0.6, 0.0, 0.0], linewidth=0.6, label='Tropopause')
    
    # plot the bottom and top of the Dentritic Growth Zone Temperature Range
    ax.axhline(y=temp_crossing_altitude(SoundingData, -10), color=[0.0, 0.8, 0.8], linewidth=0.6, label='Dentritic Growth Zone Temperature Range (-10 to -20°C)')
    ax.axhline(y=temp_crossing_altitude(SoundingData, -20), color=[0.0, 0.8, 0.8], linewidth=0.6)
    
    # plot the bottom and top of the Hallett-Mossop Temperature Range
    ax.axhline(y=temp_crossing_altitude(SoundingData, -3), color=[0.8, 0.4, 0.0], linewidth=0.6, label='Hallett-Mossop Temperature Range (-8 to -3°C)')
    ax.axhline(y=temp_crossing_altitude(SoundingData, -8), color=[0.8, 0.4, 0.0], linewidth=0.6)
    
    # plot the freezing level
    ax.axhline(y=temp_crossing_altitude(SoundingData, 0), color='blue', linewidth=0.6, label='Freezing Level (0°C)')
    
    # ax.legend(loc='upper right')
    
    # approximate latitude where the slice is taken (may vary a bit since the radar grid isn't perfect)
    ApproxLat = np.round(lat_slice[150], 2)
    
    # change it to positive and write 'north' or 'south'
    if (ApproxLat < 0):
        ApproxLat = ApproxLat * -1
        ApproxLatDir = 'S'
    else:
        ApproxLatDir = 'N'
    
    ax.set_xlabel('Longitude [Degrees East]')
    ax.set_ylabel('Altitude [km]')
    plt.title(VarName + ' (ρHV > ' + str(MinValidRhoHV) + ') Cross Section for ' + RadarSiteName + ' Radar\n For Slice Taken ' + \
          str(EWsliceKM) + ' km ' + EWsliceNorS + ' of the Radar (~' + str(ApproxLat) + '° ' + ApproxLatDir + ')\n' + \
          RadarFileDate[0:4] + '-' + RadarFileDate[4:6] + '-' + RadarFileDate[6:8] + ' at ' + \
          str(RadarFileTime)[0:2] + ':' + str(RadarFileTime)[2:4] + ':' + str(RadarFileTime)[4:6] + ' UTC with Minimum Detectable Z', pad=12)
    plt.grid(linewidth=0.5, alpha=0.3)
    ax.set_yticks(np.arange(0, MaxHeight + 3 + 1, 2))

    
    plt.xlim([np.min(lon_slice), np.max(lon_slice)])
    plt.ylim([0, MaxHeight + 3])
    
    SaveFolder = '/scratch/v46/sg3241/tmp/pngImages/' + PlotType + '/' + RadarIDno + '/' + \
                                                        RadarFileDate + '/' + VarName + '/RhoHVlim' + str(MinValidRhoHV*100)[0:2] + '/'
                                                                                                      # RhoHV 0.85 becomes 85 in file name
    SaveFile   = RadarIDno + '_' + RadarFileDate + '_' + RadarFileTime + '_' + VarNameLong + '_' + PlotType + \
    str(EWsliceKM) + EWsliceNorS + '_MinDBZ.png'
    
    SavePath = SaveFolder + SaveFile
    
    if not Path(SaveFolder).exists():
        print('Creating Folder: ' + SaveFolder)
        Path(SaveFolder).mkdir(parents=True, exist_ok=True)
    
    plt.savefig(SavePath, bbox_inches='tight', facecolor='w', dpi = 300)
    # plt.close()


In [ ]:
VarMaxVal

In [ ]:
    beam_altitude_curves = np.outer(np.tan(np.deg2rad(elevation_angles)), 
                                    horz_vals)

In [ ]:
np.tan(np.deg2rad(elevation_angles))

In [ ]:
np.shape(horz_vals)

In [ ]:
xgrid

In [ ]:
# GIF MAKER
# FOR Vertical Cross Sections

# SavedFolder = '/scratch/v46/sg3241/tmp/pngImages/Vert/' + RadarIDno + '/' + RadarFileDate + '/'
SavedFolder = '/scratch/v46/sg3241/tmp/pngImages/Vert/' + RadarIDno + '/' + RadarFileDate + '/' + VarName + '/RhoHVlim' + str(MinValidRhoHV*100)[0:2] + '/'
# 22_20240214_000000_corrected_reflectivity_Vert25North.png

# LOADING IMAGES
files = sorted(os.listdir(SavedFolder)) # takes all of the files in the folder in the order they are named

images = [Image.open(os.path.join(SavedFolder, f))
    for f in files
    if f.endswith(str(EWsliceKM) + EWsliceNorS + '_MinDBZ.png')]

# GIFsaveFolder = '/scratch/v46/sg3241/tmp/gifImages/Vert/' + RadarIDno + '/' + RadarFileDate + '/'
# GIFsaveFile   = RadarIDno + '_' + RadarFileDate + '_' + PlotVar + '_' + str(EWsliceKM) + EWsliceNorS + '.gif'
GIFsaveFolder = '/scratch/v46/sg3241/tmp/gifImages/Vert/' + RadarIDno + '/' + RadarFileDate + '/'
GIFsaveFile   = RadarIDno + '_' + RadarFileDate + '_' + VarName + '_RhoHVlim' + str(MinValidRhoHV*100)[0:2] + '_' + str(EWsliceKM) + EWsliceNorS + '_MinDBZ.gif'

GIFsavePath = GIFsaveFolder + GIFsaveFile

if not Path(GIFsaveFolder).exists():
    Path(GIFsaveFolder).mkdir(parents=True, exist_ok=True)

# Save as looping GIF
images[0].save(GIFsavePath,
    save_all=True,
    append_images=images[1:],
    duration=200,    # ms per frame
    loop=0)          # 0 = loop forever

print('Saved GIF for ' + RadarFileDate)